# 04 · H3 и сравнение методов: чинит ли steering разрыв, и не дешевле ли альтернатива

Полный цикл: база → probe → вектор на train → подбор на val → обучаемые конкуренты
(гейт, TACT, LoRA — train → выбор на val) → сравнение всех методов на val →
заморозка всего набора → test один раз (сравнения, sham-контроли, многошаговые эпизоды).

| метод | что проверяет |
|---|---|
| strong_prompt, restate, explicit_state | хватает ли промпта / пересказа / явного состояния (вердикт ALT) |
| steering | H3: CAA-вектор в окне после ответа |
| steering_gated | тот же вектор, но только когда гейт на активациях предсказывает ошибку |
| tact | адаптация TACT-Gate (2605.05980) |
| lora | H4: дообучение на тех же данных |
| sham:* | контроли H3 |

Запускать после 03, когда H1 и H2 подтверждены. Все решения базы и активации
берутся из кэша 03 — пересчитывается только новое.

> **Протокол** (`docs/HYPOTHESES.md`, `AGENTS.md`): вектор, гейт, оси TACT и LoRA строятся только
> на train, подбор — только на val, test открывается **один раз** на замороженный набор методов.
> Пайплайн это проверяет сам — если ячейка упала с `TestLockError` или `StageOrderError`,
> это защита сработала, а не баг.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="h3-steering-qwen3-1.7b", hypothesis="H3", notebook="notebooks/04_h3_steering.ipynb",
    notes="Steering (CAA) против промпта, пересказа, явного состояния, гейта, TACT и LoRA.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "eval.methods": ["baseline", "strong_prompt", "restate", "explicit_state",
                         "steering", "steering_gated", "tact", "lora"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
    },
)
exp = Experiment(cfg)
exp.data()
exp.estimate()

In [ ]:
exp.run("sweep")
viz.dose_response(exp.sweep_table(), cfg.steer.max_specificity_drop);
exp.state["selection"]

### Обучаемые конкуренты (train → выбор на val) и все методы на val

Для выбранных методов val оптимистичен: на нём же шёл подбор. Решает только test.

In [ ]:
exp.fit()
exp.results_table("val")

In [ ]:
for m in ("steering_gated", "tact", "lora"):
    t = exp.fit_table(m)
    if not t.empty:
        display(Markdown(f"**{m}** — подбор на val"))
        display(t.drop(columns=[c for c in ("path", "vector_id", "gate_id", "axes_id") if c in t]))

In [ ]:
for h, v in exp.verdicts().items():   # предварительно, по val
    print(f"{h}: {v.label:18} {v.reason}")

### Test — один раз. Дальше этой ячейки конфиг не меняется.

Открывать, только когда результаты val просмотрены и решение принято: повторное открытие для того же набора методов заблокировано, а для другого — помечается в журнале.

In [ ]:
exp.freeze()
exp.test()
viz.compare_methods(exp.results_table("test"));
exp.results_table("test")

In [ ]:
for h, v in exp.verdicts().items():
    print(f"{h}: {v.label:16} {v.reason}")

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))